# 07 — Latency Benchmarking
## Inference Time Profiling for Real-Time Fraud Detection

This notebook benchmarks inference latency for all models to assess
suitability for real-time transaction processing (<200ms target).

Produces: V27 (latency box plot), V28 (latency histogram)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import joblib

from src.evaluation.latency import (
    benchmark_latency, plot_latency_boxplot, plot_latency_histogram
)

FIGURES_DIR = Path('../outputs/figures')
TABLES_DIR = Path('../outputs/tables')
MODELS_DIR = Path('../outputs/models')
PROCESSED_DIR = Path('../data/processed')

for d in [FIGURES_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 1. Load Models and Test Data

In [ ]:
from sklearn.preprocessing import LabelEncoder

X_test = pd.read_csv(PROCESSED_DIR / 'baf_X_test.csv')
print(f'Test data: {X_test.shape}')

# Load all models
models = {}
model_files = {
    'Logistic Regression': 'logistic_regression.joblib',
    'Random Forest': 'random_forest.joblib',
    'XGBoost': 'xgboost.joblib',
    'XGB-RF Ensemble': 'xgb_rf_ensemble.joblib',
}

for name, filename in model_files.items():
    try:
        m = joblib.load(MODELS_DIR / filename)
        if hasattr(m, 'n_features_in_') and m.n_features_in_ != X_test.shape[1]:
            print(f'Skipped: {name} ({m.n_features_in_} features != {X_test.shape[1]})')
            continue
        models[name] = m
        print(f'Loaded: {name}')
    except FileNotFoundError:
        print(f'Not found: {name} ({filename})')

# Load FL ensemble — use BAF-specific models (must match BAF test features)
try:
    from sklearn.ensemble import VotingClassifier
    fl_xgb = joblib.load(MODELS_DIR / 'fl_baf_xgb.joblib')
    fl_rf = joblib.load(MODELS_DIR / 'fl_baf_rf.joblib')
    
    # Verify feature match
    n_xgb = getattr(fl_xgb, 'n_features_in_', X_test.shape[1])
    n_rf = getattr(fl_rf, 'n_features_in_', X_test.shape[1])
    if n_xgb == X_test.shape[1] and n_rf == X_test.shape[1]:
        fl_ensemble = VotingClassifier(
            estimators=[('xgb', fl_xgb), ('rf', fl_rf)],
            voting='soft',
        )
        fl_ensemble.estimators_ = [fl_xgb, fl_rf]
        le = LabelEncoder()
        le.classes_ = np.array([0, 1])
        fl_ensemble.le_ = le
        fl_ensemble.classes_ = np.array([0, 1])
        models['FL Ensemble'] = fl_ensemble
        print('Loaded: FL Ensemble (BAF)')
    else:
        print(f'FL models feature mismatch: XGB={n_xgb}, RF={n_rf}, test={X_test.shape[1]}')
except FileNotFoundError:
    print('FL BAF models not found — run notebook 05 first')
except Exception as e:
    print(f'FL Ensemble not available: {e}')

## 2. Benchmark All Models

In [ ]:
N_ITERATIONS = 5000

latency_results = {}
all_latencies = []
model_names = []

for name, model in models.items():
    print(f'\nBenchmarking {name} ({N_ITERATIONS} iterations)...')
    results, times = benchmark_latency(model, X_test, n_iterations=N_ITERATIONS)
    latency_results[name] = results
    all_latencies.append(times)
    model_names.append(name)
    
    print(f'  Mean:   {results["mean_ms"]:.3f} ms')
    print(f'  Median: {results["median_ms"]:.3f} ms')
    print(f'  P95:    {results["p95_ms"]:.3f} ms')
    print(f'  P99:    {results["p99_ms"]:.3f} ms')
    print(f'  Max:    {results["max_ms"]:.3f} ms')

## 3. V27 — Latency Box Plot

In [ ]:
plot_latency_boxplot(all_latencies, model_names, output_dir=str(FIGURES_DIR))
print('Saved: latency_boxplot.png')

## 4. V28 — Latency Histogram (FL Ensemble)

In [ ]:
# Use the last model's latencies (FL or centralized ensemble)
ensemble_name = 'FL Ensemble' if 'FL Ensemble' in latency_results else 'XGB-RF Ensemble'
ensemble_idx = model_names.index(ensemble_name)
plot_latency_histogram(all_latencies[ensemble_idx], model_name=ensemble_name,
                       output_dir=str(FIGURES_DIR))
print(f'Saved: latency_histogram_{ensemble_name.lower().replace(" ", "_")}.png')

## 5. Latency Summary Table

In [ ]:
latency_df = pd.DataFrame([
    {'Model': name, **{k: f'{v:.3f}' for k, v in results.items()}}
    for name, results in latency_results.items()
])
latency_df.to_csv(TABLES_DIR / 'latency_results.csv', index=False)
print(latency_df.to_string(index=False))

# Check against 200ms target
print('\n=== Real-Time Suitability (< 200ms P99) ===')
for name, results in latency_results.items():
    p99 = results['p99_ms']
    status = 'PASS' if p99 < 200 else 'FAIL'
    print(f'  {name}: P99 = {p99:.3f}ms [{status}]')